<a href="https://colab.research.google.com/github/di-yeferson/analitica-empresarial-integrada/blob/main/LABORATORIO_C1_AEI_DIEGO%2C_YEFERSON_2026_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LABORATORIO DIRIGIDO N.° 01
## Analítica Empresarial Integrada — TECSUP 2026-I

| | |
|---|---|
| **Docente** | Pilar Rocío Sayán Mejía |
| **N.° de laboratorio** | 1 |
| **Plan curricular** | Big Data y Ciencia de Datos · Código C28 · 2026-I |
| **Duración** | 200 minutos (6:50 p.m. – 10:10 p.m., con receso de 15 minutos) |
---

**Estudiante:** _(QUISPE HUAMANI, DIEGO YEFERSON)_ &nbsp;&nbsp;&nbsp; **Sección:** 6C28 &nbsp;&nbsp;&nbsp; **Fecha:** 30/08/26

# Caso: Andes Supply S.A.C.

Andes Supply S.A.C. es una empresa peruana que provee bienes y servicios industriales a las compañías mineras del país: repuestos, mantenimiento de equipo pesado y servicios de logística en mina. Vende a crédito y actualmente otorga a todos sus clientes las mismas condiciones: noventa días para pagar.

En 2025 dos clientes dejaron facturas impagas y la gerencia general decidió que, a partir de 2026, las condiciones de crédito dejarán de ser iguales para todos. El área comercial debe clasificar a cada cliente minero en uno de tres tramos: crédito a 90 días, crédito a 30 días o pago adelantado. Además, debe sustentar la clasificación ante el cliente.

Andes Supply no cuenta con información interna de sus clientes. Solo puede acceder a la información financiera pública que las mineras presentan ante la Superintendencia del Mercado de Valores.

**Pregunta de negocio:** ¿qué condiciones de crédito debe otorgar Andes Supply a cada cliente minero, con qué evidencia lo sustenta y qué no puede afirmar con la información que tiene?

Los estados financieros de las mineras y las cotizaciones del BCRP son datos reales, oficiales y públicos. Andes Supply es una empresa ficticia creada únicamente para contextualizar la decisión.

# Ejercicio 1 — Ficha de trazabilidad del activo de datos (2 puntos)

**Resultado exigido.** Documente el alcance real del conjunto de datos que acaba de descargar. La gerencia debe poder saber, sin abrir el código, de dónde proviene la información y hasta dónde llega.

**La entrega debe contener:**

- La fuente y el servicio oficial efectivamente consultados, y el ejercicio económico al que corresponden los estados financieros.
- La cantidad de empresas distintas que devuelve la consulta de información financiera.
- La cantidad de sectores económicos distintos presentes en esa respuesta.
- La cantidad de cuentas contables distintas que trae el estado de situación financiera.
- Las monedas en que reportan las empresas del conjunto.

**Criterio de aceptación.** Los cinco valores numéricos deben obtenerse por cálculo sobre las tablas descargadas. Si alguno aparece escrito literalmente en el código, el criterio se considera no logrado.

In [2]:
%pip install -q polars==1.17.1 duckdb==1.1.3

import json, html, re
from xml.etree import ElementTree as ET
import requests
import polars as pl
import pandas as pd          # solo para recibir la respuesta de la SMV
import plotly.express as px
from IPython.display import display

pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_width_chars(180)

SERVICIO_SMV = "https://mvnet.smv.gob.pe/ws_od_eeff/WebServiceInfoFinanciera.asmx"
EJERCICIO = 2024
SECTOR = "MINERAS"

# Codigos oficiales de cuenta del Estado de Situacion Financiera.
# Se identifican por CODIGO y no por descripcion: existen cuentas cuyo texto
# tambien contiene "Activos Corrientes" sin ser el total.
TOTAL_ACTIVO_CORRIENTE = "1D01ST"
TOTAL_PASIVO_CORRIENTE = "1D03ST"

print("polars:", pl.__version__)
print("Entorno listo.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 85.8 MB/s eta 0:00:00
polars: 1.17.1
Entorno listo.


In [3]:
def _local(etiqueta):
    return etiqueta.split("}")[-1]

def _a_dataframe(payload):
    payload = html.unescape(payload).strip()
    try:
        objeto = json.loads(payload)
        if isinstance(objeto, dict): objeto = objeto.get("rows", objeto.get("data", objeto))
        if isinstance(objeto, dict): objeto = [objeto]
        return pd.DataFrame(objeto)
    except json.JSONDecodeError:
        raiz = ET.fromstring(payload); registros = []
        for nodo in raiz.iter():
            hijos = list(nodo)
            if len(hijos) >= 5 and all(not list(h) for h in hijos):
                registros.append({_local(h.tag): h.text for h in hijos})
        return pd.DataFrame(registros)

def descargar_smv(operacion, ejercicio, periodo="A", tipo="I"):
    sobre = f"""<?xml version="1.0" encoding="utf-8"?>
    <soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
      <soap:Body><{operacion} xmlns="http://tempuri.org/">
        <Ejercicio>{ejercicio}</Ejercicio><Periodo>{periodo}</Periodo><Tipo>{tipo}</Tipo>
      </{operacion}></soap:Body></soap:Envelope>"""
    respuesta = requests.post(
        SERVICIO_SMV, data=sobre.encode("utf-8"), timeout=180,
        headers={"Content-Type": "text/xml; charset=utf-8",
                 "SOAPAction": f'"http://tempuri.org/{operacion}"'})
    respuesta.raise_for_status()
    raiz = ET.fromstring(respuesta.content)
    resultado = next((n for n in raiz.iter() if _local(n.tag) == f"{operacion}Result"), None)
    if resultado is None or not (resultado.text or "").strip():
        raise ValueError(f"La SMV no devolvio datos para {operacion}.")
    return pl.from_pandas(_a_dataframe(resultado.text))

principales = descargar_smv("obtener_InfoFinanciera", EJERCICIO)
balance = descargar_smv("obtener_BalanceGeneral", EJERCICIO)

print("Cuentas principales      :", principales.shape)
print("Estado de situacion fin. :", balance.shape)
display(principales.head(3))

Cuentas principales      : (276, 16)
Estado de situacion fin. : (20574, 15)


RPJ,TipoEmpresa,TipoSector,NombreEmpresa,RUC,CIIU,Ejercicio,TipoInformacion,Trimestre,Moneda,MetodoFlujoEfectivo,ActivoTotal,PatrimonioTotal,TotalIngreso,UtilidadNeta,PasivoTotal
str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64
"""L00474""","""EMPRESAS MERCADO ALTERNATIVO D…","""DIVERSOS""","""A. JAIME ROJAS REPRESENTACIONE…","""20102032951""","""5190""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Directo""",77710,39497,77196,6786,38213
"""I00004""","""SOCIEDADES ADMINISTRADORAS DE …","""""","""AC CAPITALES SOCIEDAD ADMINIST…","""20504893295""","""6430""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Directo""",9070,7373,4880,-85,1697
"""OE7511""","""SOCIEDADES ADMINISTRADORAS DE …","""""","""ACRES SOCIEDAD ADMINISTRADORA …","""20601498996""","""6430""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Indirecto""",2724,2238,2348,176,486


In [4]:
# SOLUCION | Ejercicio 1 (2 puntos) — Ficha de trazabilidad
# Los cinco valores se obtienen por calculo sobre las tablas descargadas.
# Ninguno esta escrito a mano: ese es el criterio de aceptacion del ejercicio.
FICHA = {
    "fuente":             "SMV - Portal de Datos Abiertos",
    "servicio":           SERVICIO_SMV,
    "ejercicio":          EJERCICIO,
    "empresas_totales":   principales["NombreEmpresa"].n_unique(),
    "sectores_distintos": principales["TipoSector"].n_unique(),
    "cuentas_balance":    balance["Cuenta"].n_unique(),
    "monedas":            sorted(principales["Moneda"].unique().to_list()),
}

for k, v in FICHA.items():
    print(f"{k:>20}: {v}")

# Error frecuente: contar filas en lugar de valores distintos. `principales`
# trae varias filas por empresa (una por tipo de informacion), de modo que
# principales.height NO es la cantidad de empresas.
print("\nFilas de principales      :", principales.height)
print("Empresas distintas        :", principales["NombreEmpresa"].n_unique())


              fuente: SMV - Portal de Datos Abiertos
            servicio: https://mvnet.smv.gob.pe/ws_od_eeff/WebServiceInfoFinanciera.asmx
           ejercicio: 2024
    empresas_totales: 276
  sectores_distintos: 10
     cuentas_balance: 482
             monedas: ['D lares', 'Soles']

Filas de principales      : 276
Empresas distintas        : 276


# Ejercicio 2 — Delimitación del sector evaluable (2 puntos)

**Resultado exigido.** Obtenga la tabla de empresas mineras sobre la que se construirá todo el análisis posterior. No todas las empresas que devuelve el servicio son analizables: algunas pertenecen a otros sectores, otras reportan periodos que no son anuales y otras presentan cuentas incompletas.

**La entrega debe contener:**

- Solo empresas del sector minero.
- Solo información de periodicidad anual.
- Las cinco magnitudes del análisis —activo total, patrimonio, ingresos, utilidad neta y pasivo total— expresadas como valores numéricos.
- Sin empresas que presenten alguna de esas cinco magnitudes vacía.
- Sin empresas con ingresos iguales a cero, porque impiden calcular el margen.

**Criterio de aceptación.** El notebook debe informar cuántas empresas quedaron disponibles tras la delimitación. Una tabla que conserve el total de empresas descargadas indica que los criterios no se aplicaron.

In [5]:
# SOLUCION  (2 puntos) — Delimitacion del sector evaluable

print(principales["TipoInformacion"].unique().to_list())

MAGNITUDES = ["ActivoTotal", "PatrimonioTotal", "TotalIngreso", "UtilidadNeta", "PasivoTotal"]

mineras = (
    principales
    .filter(pl.col("TipoSector") == SECTOR)
    .filter(pl.col("TipoInformacion").str.to_uppercase().str.contains("ANUAL"))
    .with_columns([pl.col(c).cast(pl.Float64, strict=False) for c in MAGNITUDES])
    .filter(pl.all_horizontal([pl.col(c).is_not_null() for c in MAGNITUDES]))
    .filter(pl.col("TotalIngreso") != 0)
)

print("Empresas mineras descargadas (sin filtrar) :", principales.filter(pl.col("TipoSector") == SECTOR).height)
print("Empresas mineras evaluables tras delimitar :", mineras.height)
display(mineras.select(["NombreEmpresa", "Moneda"] + MAGNITUDES))

['Anual Individual']
Empresas mineras descargadas (sin filtrar) : 16
Empresas mineras evaluables tras delimitar : 14


NombreEmpresa,Moneda,ActivoTotal,PatrimonioTotal,TotalIngreso,UtilidadNeta,PasivoTotal
str,str,f64,f64,f64,f64,f64
"""COMPAÑIA DE MINAS BUENAVENTURA…","""D lares""",4.42137e6,3.390689e6,665978.0,402689.0,1.030681e6
"""COMPAÑIA MINERA PODEROSA S.A.A…","""Soles""",2.542791e6,1.883522e6,2.624541e6,415128.0,659269.0
"""COMPAÑIA MINERA SANTA LUISA S.…","""Soles""",523486.0,344321.0,467521.0,102749.0,179165.0
"""MINERA ANDINA DE EXPLORACIONES…","""Soles""",14050.0,6743.0,5370.0,-181.0,7307.0
"""MINSUR S.A.""","""D lares""",2.499941e6,1.625737e6,1.038312e6,463546.0,874204.0
"""NEXA RESOURCES ATACOCHA S.A.A.""","""D lares""",107814.0,8459.0,93382.0,10133.0,99355.0
"""NEXA RESOURCES PERU S.A.A.""","""D lares""",1.066459e6,749906.0,553200.0,-15524.0,316553.0
"""PERUBAR S.A.""","""D lares""",89581.0,68020.0,24021.0,2312.0,21561.0
"""SHOUGANG HIERRO PERU S.A.A.""","""Soles""",1.0575844e7,3.793098e6,5.864032e6,2.089269e6,6.782746e6


# Ejercicio 3 — Perfil financiero comparado del sector (2 puntos)

**Resultado exigido.** Construya el perfil que permitirá comparar a los clientes entre sí. Se exigen seis indicadores por empresa: margen neto, rentabilidad sobre activos, rentabilidad sobre patrimonio, razón corriente, razón de endeudamiento y apalancamiento.

**La entrega debe contener:**

- El activo corriente y el pasivo corriente deben extraerse del estado de situación financiera identificando la cuenta por su código oficial y no por su descripción textual.
- Los seis indicadores deben quedar incorporados como columnas del perfil, no impresos sueltos.
- El resultado debe presentarse ordenado y legible, con el nombre de la empresa y su moneda.

**Criterio de aceptación.** Ninguna razón puede resultar igual a cero ni infinita. Si la razón corriente sale cero, la cuenta seleccionada no es la correcta: existen cuentas cuya descripción contiene el texto «Activos Corrientes» sin ser el total, y valen cero.

In [6]:
# SOLUCION | Perfil financiero comparado del sector


activo_corriente = (
    balance.filter(pl.col("Cuenta") == TOTAL_ACTIVO_CORRIENTE)
    .select(["RPJ", pl.col("Monto1").cast(pl.Float64, strict=False).alias("ActivoCorriente")])
)
pasivo_corriente = (
    balance.filter(pl.col("Cuenta") == TOTAL_PASIVO_CORRIENTE)
    .select(["RPJ", pl.col("Monto1").cast(pl.Float64, strict=False).alias("PasivoCorriente")])
)

perfil = (
    mineras
    .join(activo_corriente, on="RPJ", how="inner")
    .join(pasivo_corriente, on="RPJ", how="inner")
    .with_columns([
        (pl.col("UtilidadNeta") / pl.col("TotalIngreso")).alias("MargenNeto"),
        (pl.col("UtilidadNeta") / pl.col("ActivoTotal")).alias("ROA"),
        (pl.col("UtilidadNeta") / pl.col("PatrimonioTotal")).alias("ROE"),
        (pl.col("ActivoCorriente") / pl.col("PasivoCorriente")).alias("RazonCorriente"),
        (pl.col("PasivoTotal") / pl.col("ActivoTotal")).alias("Endeudamiento"),
        (pl.col("ActivoTotal") / pl.col("PatrimonioTotal")).alias("Apalancamiento"),
    ])
    # Descarta razones en cero o infinitas: si aparecen, la cuenta de
    # activo/pasivo corriente seleccionada no es la correcta.
    .filter(
        pl.col("RazonCorriente").is_finite() & (pl.col("RazonCorriente") > 0) &
        pl.col("Endeudamiento").is_finite() & (pl.col("Endeudamiento") > 0) &
        pl.col("Apalancamiento").is_finite() & (pl.col("Apalancamiento") > 0)
    )
)

print("Empresas en el perfil final:", perfil.height)
display(
    perfil.select(["NombreEmpresa", "Moneda", "MargenNeto", "ROA", "ROE",
                    "RazonCorriente", "Endeudamiento", "Apalancamiento"])
    .sort("NombreEmpresa")
)

Empresas en el perfil final: 14


NombreEmpresa,Moneda,MargenNeto,ROA,ROE,RazonCorriente,Endeudamiento,Apalancamiento
str,str,f64,f64,f64,f64,f64,f64
"""COMPAÑIA DE MINAS BUENAVENTURA…","""D lares""",0.604658,0.091078,0.118763,1.521087,0.233113,1.303974
"""COMPAÑIA MINERA PODEROSA S.A.A…","""Soles""",0.158172,0.163257,0.2204,0.81319,0.25927,1.350019
"""COMPAÑIA MINERA SANTA LUISA S.…","""Soles""",0.219774,0.196278,0.29841,3.666625,0.342254,1.520343
"""MINERA ANDINA DE EXPLORACIONES…","""Soles""",-0.033706,-0.012883,-0.026843,2.126689,0.520071,2.083642
"""MINSUR S.A.""","""D lares""",0.446442,0.185423,0.28513,1.481281,0.34969,1.537728
"""NEXA RESOURCES ATACOCHA S.A.A.""","""D lares""",0.108511,0.093986,1.197896,1.331592,0.921541,12.745478
"""NEXA RESOURCES PERU S.A.A.""","""D lares""",-0.028062,-0.014557,-0.020701,2.585725,0.296826,1.422124
"""PERUBAR S.A.""","""D lares""",0.096249,0.025809,0.03399,0.740964,0.240687,1.31698
"""SHOUGANG HIERRO PERU S.A.A.""","""Soles""",0.356285,0.197551,0.550808,2.584993,0.641343,2.788181


# Ejercicio 4 — Lectura crítica del ranking (2 puntos)

**Resultado exigido.** Identifique a la empresa que encabeza el sector por rentabilidad sobre patrimonio y extraiga, para esa misma empresa, su endeudamiento, su apalancamiento, su patrimonio y su pasivo total.

**La entrega debe contener:**

- La empresa debe quedar determinada por el propio cálculo sobre el perfil.
- Las cuatro magnitudes deben corresponder a la empresa identificada, obtenidas de la misma tabla.

**Criterio de aceptación.** El nombre de la empresa no puede aparecer escrito en el código. Si el sector cambiara de composición, la respuesta debería actualizarse sola al reejecutar.

In [7]:
# SOLUCION - Lectura critica del ranking


lider_roe = perfil.sort("ROE", descending=True).head(1)

nombre_lider        = lider_roe["NombreEmpresa"][0]
roe_lider            = lider_roe["ROE"][0]
endeudamiento_lider  = lider_roe["Endeudamiento"][0]
apalancamiento_lider = lider_roe["Apalancamiento"][0]
patrimonio_lider     = lider_roe["PatrimonioTotal"][0]
pasivo_lider         = lider_roe["PasivoTotal"][0]

print(f"Empresa con mayor ROE del sector : {nombre_lider}")
print(f"ROE                              : {roe_lider:.2%}")
print(f"Endeudamiento                    : {endeudamiento_lider:.2%}")
print(f"Apalancamiento                   : {apalancamiento_lider:.2f}")
print(f"Patrimonio total                 : {patrimonio_lider:,.0f}")
print(f"Pasivo total                     : {pasivo_lider:,.0f}")

Empresa con mayor ROE del sector : NEXA RESOURCES ATACOCHA S.A.A.
ROE                              : 119.79%
Endeudamiento                    : 92.15%
Apalancamiento                   : 12.75
Patrimonio total                 : 8,459
Pasivo total                     : 99,355


# Ejercicio 5 — Política de crédito de Andes Supply (2 puntos)

**Resultado exigido.** Clasifique a cada cliente minero en uno de tres tramos de condiciones comerciales: crédito a noventa días, crédito a treinta días o pago adelantado. Esta es la decisión que la gerencia va a aplicar.

**La entrega debe contener:**

- Los umbrales que separan los tramos los define su equipo, pero deben aparecer declarados como valores explícitos y localizables, no incrustados dentro de la lógica.
- La clasificación debe emplear al menos dos indicadores distintos; un solo indicador no sustenta una política de crédito.
- El resultado debe mostrar la clasificación por empresa y el recuento de clientes en cada tramo.

**Criterio de aceptación.** Cada umbral debe poder justificarse ante un cliente que reclame su clasificación. Umbrales elegidos sin criterio explicable se califican como no logrados aunque el código funcione.

In [8]:
# SOLUCION |  Politica de credito de Andes Supply

UMBRAL_ENDEUDAMIENTO_BAJO = 0.40   # hasta aqui, la empresa financia sus activos sobre todo con patrimonio propio
UMBRAL_ENDEUDAMIENTO_ALTO = 0.60   # por encima de esto, mas del 60% del activo esta financiado con deuda de terceros
UMBRAL_LIQUIDEZ_BUENA     = 1.50   # por cada sol de deuda de corto plazo tiene 1.5 soles de activo corriente
UMBRAL_LIQUIDEZ_MINIMA    = 1.00   # por debajo de 1, el activo corriente no alcanza a cubrir el pasivo corriente

def clasificar(endeudamiento, liquidez):
    if endeudamiento <= UMBRAL_ENDEUDAMIENTO_BAJO and liquidez >= UMBRAL_LIQUIDEZ_BUENA:
        return "Credito a 90 dias"
    if endeudamiento <= UMBRAL_ENDEUDAMIENTO_ALTO and liquidez >= UMBRAL_LIQUIDEZ_MINIMA:
        return "Credito a 30 dias"
    return "Pago adelantado"

politica = perfil.with_columns(
    pl.struct(["Endeudamiento", "RazonCorriente"])
    .map_elements(lambda r: clasificar(r["Endeudamiento"], r["RazonCorriente"]), return_dtype=pl.Utf8)
    .alias("TramoCredito")
)

display(politica.select(["NombreEmpresa", "Endeudamiento", "RazonCorriente", "TramoCredito"]).sort("TramoCredito"))
print(politica.group_by("TramoCredito").agg(pl.len().alias("Clientes")))

NombreEmpresa,Endeudamiento,RazonCorriente,TramoCredito
str,f64,f64,str
"""MINERA ANDINA DE EXPLORACIONES…",0.520071,2.126689,"""Credito a 30 dias"""
"""MINSUR S.A.""",0.34969,1.481281,"""Credito a 30 dias"""
"""VOLCAN COMPAÑIA MINERA S.A.A.""",0.581698,1.592866,"""Credito a 30 dias"""
"""COMPAÑIA DE MINAS BUENAVENTURA…",0.233113,1.521087,"""Credito a 90 dias"""
"""COMPAÑIA MINERA SANTA LUISA S.…",0.342254,3.666625,"""Credito a 90 dias"""
"""NEXA RESOURCES PERU S.A.A.""",0.296826,2.585725,"""Credito a 90 dias"""
"""SOCIEDAD MINERA CERRO VERDE S.…",0.155701,3.556434,"""Credito a 90 dias"""
"""SOCIEDAD MINERA CORONA S.A.""",0.345655,1.529011,"""Credito a 90 dias"""
"""SOCIEDAD MINERA EL BROCAL S.A.…",0.389274,1.825253,"""Credito a 90 dias"""


shape: (3, 2)
┌───────────────────┬──────────┐
│ TramoCredito      ┆ Clientes │
│ ---               ┆ ---      │
│ str               ┆ u32      │
╞═══════════════════╪══════════╡
│ Credito a 30 dias ┆ 3        │
│ Credito a 90 dias ┆ 7        │
│ Pago adelantado   ┆ 4        │
└───────────────────┴──────────┘


# Ejercicio 6 — Consulta reproducible de la cartera en SQL (2 puntos)

**Resultado exigido.** La cartera resultante debe poder consultarse por personas que leen SQL y no Python. Formule sobre el perfil una consulta que devuelva las empresas con razón corriente igual o mayor que uno y endeudamiento inferior a 0,60, ordenadas de mayor a menor rentabilidad sobre patrimonio.

**La entrega debe contener:**

- Al menos cuatro columnas en la selección.
- Dos condiciones de filtrado enlazadas entre sí.
- Una columna calculada que etiquete el nivel de riesgo según las condiciones que su equipo defina.
- Ordenamiento explícito del resultado.

**Criterio de aceptación.** El resultado debe contrastarse con la clasificación construida en el ejercicio anterior. La coincidencia o discrepancia entre ambos caminos es lo que se interpreta en la pregunta escrita 6.1.

In [9]:
# SOLUCION  — Consulta reproducible de la cartera en SQL

import duckdb

cartera_sql = duckdb.sql(f"""
    SELECT
        NombreEmpresa,
        RazonCorriente,
        Endeudamiento,
        ROE,
        CASE
            WHEN Endeudamiento < {UMBRAL_ENDEUDAMIENTO_BAJO} THEN 'Riesgo bajo'
            WHEN Endeudamiento < {UMBRAL_ENDEUDAMIENTO_ALTO} THEN 'Riesgo medio'
            ELSE 'Riesgo alto'
        END AS NivelRiesgo
    FROM perfil
    WHERE RazonCorriente >= 1 AND Endeudamiento < {UMBRAL_ENDEUDAMIENTO_ALTO}
    ORDER BY ROE DESC
""").pl()

display(cartera_sql)

NombreEmpresa,RazonCorriente,Endeudamiento,ROE,NivelRiesgo
str,f64,f64,f64,str
"""COMPAÑIA MINERA SANTA LUISA S.…",3.666625,0.342254,0.29841,"""Riesgo bajo"""
"""MINSUR S.A.""",1.481281,0.34969,0.28513,"""Riesgo bajo"""
"""SOUTHERN PERU COPPER CORPORATI…",2.436788,0.208547,0.227934,"""Riesgo bajo"""
"""SOCIEDAD MINERA CERRO VERDE S.…",3.556434,0.155701,0.140514,"""Riesgo bajo"""
"""VOLCAN COMPAÑIA MINERA S.A.A.""",1.592866,0.581698,0.136962,"""Riesgo medio"""
"""COMPAÑIA DE MINAS BUENAVENTURA…",1.521087,0.233113,0.118763,"""Riesgo bajo"""
"""SOCIEDAD MINERA EL BROCAL S.A.…",1.825253,0.389274,0.084933,"""Riesgo bajo"""
"""NEXA RESOURCES PERU S.A.A.""",2.585725,0.296826,-0.020701,"""Riesgo bajo"""
"""MINERA ANDINA DE EXPLORACIONES…",2.126689,0.520071,-0.026843,"""Riesgo medio"""


# Preguntas escritas de interpretación

Las preguntas escritas no otorgan puntaje separado: son la evidencia con la que se califica la interpretación dentro de cada criterio de la rúbrica. Un notebook que ejecuta correctamente pero no interpreta no alcanza el nivel Excelente en ningún criterio.

El criterio «Escalera analítica» se evalúa únicamente con la pregunta escrita 2.1, que no tiene ejercicio de código asociado.

## Pregunta 1.1 — Qué preguntas de negocio permite y no permite responder el conjunto de datos

**Sí permite:** identificar empresas con utilidad neta negativa en 2024; comparar rentabilidad, liquidez y estructura de financiamiento; estimar qué empresa tiene mayor capacidad contable de cubrir pasivos corrientes; y medir qué empresa depende más del financiamiento de terceros.

**No permite:** saber cuáles empresas son realmente clientes de Andes Supply, cuánto compró cada una, si paga tarde, si incumplirá en el futuro o si tiene intención de pagar. Para responder eso se necesita el historial interno de ventas, facturas, vencimientos, pagos, mora, garantías y exposición crediticia por cliente.

## Pregunta 2.1 — Clasificación de seis preguntas en los escalones de la escalera analítica

| Pregunta | Escalón | Justificación |
|---|---|---|
| ¿Cuál fue la utilidad neta de cada cliente en 2024? | Descriptiva | Resume un resultado ya observado. |
| ¿Por qué cayó la rentabilidad de los clientes de cobre? | Diagnóstica | Busca explicar las causas del resultado. |
| ¿Qué probabilidad hay de que este cliente deje de pagar? | Predictiva | Estima la probabilidad de un evento futuro. |
| ¿A qué clientes debo exigir pago adelantado? | Prescriptiva | Recomienda una acción comercial. |
| ¿Cuánto endeudamiento tiene cada cliente? | Descriptiva | Mide una situación financiera observada. |
| ¿Qué pasaría con mi cartera si el cobre cae 20 %? | Predictiva | Proyecta el resultado bajo un escenario futuro. |

El laboratorio trabaja directamente en el escalón descriptivo y formula una regla prescriptiva preliminar. No demuestra causas ni genera una predicción validada porque solo dispone de un corte financiero, no de historial de pagos ni de una variable objetivo de incumplimiento.

## Pregunta 3.1 — Qué comparaciones invalida la convivencia de dos monedas en el sector

No es válido comparar directamente el ingreso de Cerro Verde con el de Shougang ni sumar los activos de todas las empresas, porque los montos mezclan dólares y soles. Esas operaciones requieren convertir primero a una moneda común con un criterio de tipo de cambio consistente. Sí pueden compararse el margen neto y la razón corriente entre empresas de monedas distintas, porque son razones adimensionales calculadas con importes expresados en la misma moneda dentro de cada empresa.

## Pregunta 4.1 — Por qué el ROE más alto del sector no identifica al cliente más sólido



 Atacocha S.A.A. encabeza el sector con un ROE de 119.79%, pero eso no la convierte en la cliente más sólida. Su patrimonio es de apenas 8,459 miles frente a un pasivo total de 99,355 miles: el 92.15% de su activo está financiado con deuda de terceros, y su apalancamiento es de 12.75 veces. El ROE es una razón entre utilidad y patrimonio, así que si el patrimonio es muy pequeño, el ratio se dispara aunque la utilidad no sea especialmente grande en términos absolutos. Por eso una empresa con ROE alto puede ser, al mismo tiempo, la más expuesta financieramente  que es justo lo que confirma su propia clasificación del Ejercicio 5, donde quedó en el tramo de (Pago adelantado), el más estricto de los tres. Un ROE alto mide rentabilidad sobre lo que aportaron los dueños, no la capacidad de la empresa de responder por sus deudas.

## Código de apoyo para la pregunta 5.1 — Cotizaciones oficiales del BCRP

In [ ]:
SERIES = {"PN01652XM": "Cobre (cUS$/lb)", "PN01654XM": "Oro (US$/oz)",
          "PN01653XM": "Estanio (cUS$/lb)", "PN01655XM": "Plata (US$/oz)"}
BASE_BCRP = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api"



Observaciones traidas del BCRP: 96


serie,ene_2023,dic_2024,var_pct
str,f64,f64,f64
"""Oro (US$/oz)""",1894.068182,2638.559091,39.3
"""Plata (US$/oz)""",23.752636,30.466636,28.3
"""Estanio (cUS$/lb)""",1270.040595,1308.213486,3.0
"""Cobre (cUS$/lb)""",406.296088,404.30647,-0.5


## Pregunta 5.1 — Relación entre la cotización de metales y el resultado de dos empresas, sin afirmar causalidad



Entre enero 2023 y diciembre 2024, las cotizaciones del BCRP muestran trayectorias muy distintas entre metales: el oro subió 39.3%, la plata 28.3%, el estaño 3.0%, mientras que el cobre se mantuvo prácticamente plano (-0.5%).

Cruzando esto con dos empresas del perfil: Southern Peru Copper Corporation, cuyo negocio principal es el cobre, registra un ROE de 22.79% pese a que el precio del cobre no tuvo tendencia alcista en el periodo. Compañía de Minas Buenaventura, orientada a oro y plata, tiene un ROE de 11.88% —menor que el de Southern Peru— a pesar de que los precios de ambos metales subieron con fuerza (39.3% y 28.3%).

Esto muestra que no hay una relación directa entre la subida del precio del metal y el resultado de la empresa que lo produce: la empresa expuesta al metal que más se revalorizó (Buenaventura) no es la que mostró mejor rentabilidad, y la que sí tuvo buen ROE (Southern Peru) está expuesta a un metal cuyo precio no subió.
La cotización internacional es solo uno de varios factores costos, producción, gestión, apalancamiento que explican el resultado financiero de una empresa.


## Pregunta 6.1 — Contraste entre la cartera obtenida con SQL y la clasificación construida con Polars



La consulta SQL devuelve exactamente 10 de las 14 empresas del perfil, y esas 10 coinciden uno a uno con las que en el Ejercicio 5 quedaron clasificadas en Crédito a 90 días o Crédito a 30 días: Minera Andina de Exploraciones, Minsur, Volcán, Buenaventura, Santa Luisa, Nexa Resources Perú, Cerro Verde, Corona, El Brocal y Southern Peru Copper Corporation.

 Las 4 empresas que el SQL excluye  Poderosa, Nexa Resources Atacocha, Perubar y Shougang Hierro Perú  son justamente las que el Ejercicio 5 mandó a "Pago adelantado". Esto tiene sentido porque el filtro de la consulta (RazonCorriente >= 1 AND Endeudamiento < 0.60) usa el mismo corte de endeudamiento y liquidez mínima que separa pago adelantado del resto de tramos en la función clasificar(); la única diferencia es que la consulta SQL no distingue entre 90 y 30 días, porque el enunciado del Ejercicio 6 no lo pedía. La coincidencia entre ambos caminos Polars y SQL confirma que la lógica de la política de crédito es consistente sin importar la herramienta con la que se consulte.

# Informe ejecutivo A a F

## A. Política recomendada

## B. Explicación al cliente


## C. Estadio de madurez de Andes Supply


## D. Modelo DELTA Plus en siete dimensiones



## E. Límites del análisis


## F. Iniciativa priorizada



### **Declaración de uso de asistentes de IA**

Se utilizó Claude como apoyo para entender los códigos y resolver los ejercicios.
 DE esat forma ayudando a entender y comprender cada línea  de codigo antes de incorporarla al notebook.